In [1]:
import torch
import torch.nn as nn
import tiktoken as tk
from torch.utils.data import Dataset, DataLoader

also implement to see how much each embedding query and key is rotated from their token embeddings

and test whether two words with same number of words in between give similar dot product at different time steps

In [2]:
2 * torch.arange(0, 12//2)

tensor([ 0,  2,  4,  6,  8, 10])

In [3]:
torch.arange(0, 12, 2)[:12//2]

tensor([ 0,  2,  4,  6,  8, 10])

In [4]:
x = torch.rand(50)

print(x)

x1 = x[::2]
x2 = x[1::2]

print(x1)
print(-x2)

y = torch.stack([-x2,x1], dim=-1).flatten()
y

tensor([0.0526, 0.7806, 0.3684, 0.4663, 0.5131, 0.9346, 0.3812, 0.1942, 0.4639,
        0.3505, 0.0022, 0.0080, 0.4512, 0.8804, 0.5520, 0.6578, 0.4637, 0.7933,
        0.2925, 0.3288, 0.8274, 0.6112, 0.0570, 0.9691, 0.1049, 0.2368, 0.3286,
        0.7567, 0.3073, 0.5039, 0.9841, 0.6351, 0.5355, 0.2053, 0.6289, 0.0635,
        0.9635, 0.5066, 0.0353, 0.9711, 0.7460, 0.9177, 0.8542, 0.1695, 0.4156,
        0.1176, 0.6267, 0.5564, 0.3525, 0.5536])
tensor([0.0526, 0.3684, 0.5131, 0.3812, 0.4639, 0.0022, 0.4512, 0.5520, 0.4637,
        0.2925, 0.8274, 0.0570, 0.1049, 0.3286, 0.3073, 0.9841, 0.5355, 0.6289,
        0.9635, 0.0353, 0.7460, 0.8542, 0.4156, 0.6267, 0.3525])
tensor([-0.7806, -0.4663, -0.9346, -0.1942, -0.3505, -0.0080, -0.8804, -0.6578,
        -0.7933, -0.3288, -0.6112, -0.9691, -0.2368, -0.7567, -0.5039, -0.6351,
        -0.2053, -0.0635, -0.5066, -0.9711, -0.9177, -0.1695, -0.1176, -0.5564,
        -0.5536])


tensor([-0.7806,  0.0526, -0.4663,  0.3684, -0.9346,  0.5131, -0.1942,  0.3812,
        -0.3505,  0.4639, -0.0080,  0.0022, -0.8804,  0.4512, -0.6578,  0.5520,
        -0.7933,  0.4637, -0.3288,  0.2925, -0.6112,  0.8274, -0.9691,  0.0570,
        -0.2368,  0.1049, -0.7567,  0.3286, -0.5039,  0.3073, -0.6351,  0.9841,
        -0.2053,  0.5355, -0.0635,  0.6289, -0.5066,  0.9635, -0.9711,  0.0353,
        -0.9177,  0.7460, -0.1695,  0.8542, -0.1176,  0.4156, -0.5564,  0.6267,
        -0.5536,  0.3525])

In [71]:
def rotary_emb(head_dim, context_length, device, theta_base=10000):
  assert head_dim % 2 == 0, "head_dim should be divisible by 2"

  freq = 1 / theta_base ** (torch.arange(0, head_dim, 2, device=device)[:(head_dim//2)].float()/head_dim)
  positions = torch.arange(0, context_length, device=device) # (context_length,)
  angles = positions.unsqueeze(1) * freq.unsqueeze(0) # (context_length, head_dim//2)
  angles = torch.cat([angles, angles], dim=1) # (context_length, head_dim)

  cos = torch.cos(angles)
  sin = torch.sin(angles)

  return cos, sin

def apply_rotation(query, key, cos, sin):
  # query and key shape (batch_size, num_heads, seq_len, head_dim)
  batch_size, num_heads, seq_len, head_dim = query.shape
  assert head_dim % 2 == 0, "head_dim should be divisible by 2"
  cos = cos[:seq_len,:]
  sin = sin[:seq_len,:]
  cos = cos.unsqueeze(0).unsqueeze(0)
  sin = sin.unsqueeze(0).unsqueeze(0)
  cos = cos.view(1, 1, seq_len, 2, head_dim//2).transpose(-2, -1).flatten(start_dim=-2)
  sin = sin.view(1, 1, seq_len, 2, head_dim//2).transpose(-2, -1).flatten(start_dim=-2)

  q1 = query[..., ::2]
  q2 = query[..., 1::2]
  query_2 = torch.stack([-q2, q1], dim=-1).flatten(start_dim=-2)
  rotated_query = query * cos + query_2 * sin

  k1 = key[..., ::2]
  k2 = key[..., 1::2]
  key_2 = torch.stack([-k2, k1], dim=-1).flatten(start_dim=-2)
  rotated_key = key * cos + key_2 * sin

  return rotated_query, rotated_key

In [48]:
GPT_CONFIG_124M = {
    'emb_size':768,
    'context_length':256,
    'vocab_size':50257,
    'num_heads':12,
    'num_layers':12,
    'drop_rate':0.1,
    'qkv_bias':False,
}

In [7]:
#lets implement compact class for normalization
class LayerNorm(nn.Module):
  def __init__(self, emb_size):
    super().__init__()
    self.eps = 1e-5
    self.scale = nn.Parameter(torch.ones(emb_size))
    self.shift = nn.Parameter(torch.zeros(emb_size))

  def forward(self, x):
    mean = x.mean(keepdim=True, dim=-1)
    variance = x.var(keepdim=True, dim=-1, unbiased=False)
    norm_x = (x - mean)/torch.sqrt(variance + self.eps)
    return self.scale * norm_x + self.shift

In [8]:
class GeLU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return 0.5 * x * (1 + torch.tanh(
        torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x,3))
    ))

In [9]:
class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg['emb_size'], 4 * cfg['emb_size']),
        GeLU(),
        nn.Linear(4 * cfg['emb_size'], cfg['emb_size'])
    )

  def forward(self, x):
    return self.layers(x)

In [72]:
class MultiHeadAttention(nn.Module):
  def __init__(self, dim_in, dim_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    assert (dim_out % num_heads == 0), "dim_out must be divisible by num_heads"

    self.dim_out = dim_out # final merged context vector embedding size
    self.num_heads = num_heads
    self.head_dim = dim_out//num_heads # embedding size of context vector in single head
    self.w_query = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
    self.w_key = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
    self.w_value = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
    self.out_proj = torch.nn.Linear(dim_out, dim_out) # transform merged context_vectors into similar dimension size vectors
    self.dropout = torch.nn.Dropout(dropout)
    self.register_buffer(
        'mask',
        torch.triu(torch.ones(context_length, context_length), diagonal=1)
    )

  def forward(self, x):
    batch_size, num_tokens, dim_in = x.shape
    queries = self.w_query(x)
    keys = self.w_key(x)
    values = self.w_value(x)  #shape (batch_size, num_tokens, dim_out)

    queries = queries.view(batch_size, num_tokens, self.num_heads, self.head_dim)
    keys = keys.view(batch_size, num_tokens, self.num_heads, self.head_dim)
    values = values.view(batch_size, num_tokens, self.num_heads, self.head_dim) #shape (batch_size, num_tokens, num_heads, head_dim)

    queries = queries.transpose(1,2)
    keys = keys.transpose(1,2)
    values = values.transpose(1,2) #shape (batch_size, num_heads, num_tokens, head_dim)

    # RoPE
    cos, sin = rotary_emb(self.head_dim, num_tokens, device=x.device)
    queries, keys = apply_rotation(queries, keys, cos, sin)

    attention_scores = queries @ keys.transpose(2,3)
    attention_scores.masked_fill_(self.mask.bool()[:num_tokens,:num_tokens], -torch.inf)

    attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim=-1)
    attention_weights = self.dropout(attention_weights)

    context_vectors = (attention_weights @ values).transpose(1,2) #transposing axis 1,2  since we have to merge the context vectors by num_heads and head_dim, so required shape will now be (batch_size, num_tokens, num_heads, head_dim)
    context_vectors = context_vectors.contiguous().view(batch_size, num_tokens, self.dim_out)

    context_vectors = self.out_proj(context_vectors)

    return context_vectors

In [11]:
class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.mha = MultiHeadAttention(cfg['emb_size'], cfg['emb_size'], cfg['context_length'], cfg['drop_rate'], cfg['num_heads'], qkv_bias=cfg['qkv_bias'])
    self.layer_norm1 = LayerNorm(cfg['emb_size'])
    self.layer_norm2 = LayerNorm(cfg['emb_size'])
    self.ffn = FeedForward(cfg)
    self.dropout = nn.Dropout(cfg['drop_rate'])

  def forward(self, x):
    shortcut = x
    x = self.layer_norm1(x)
    x = self.mha(x)
    x = self.dropout(x)
    x = x + shortcut

    shortcut = x
    x = self.layer_norm2(x)
    x = self.ffn(x)
    x = self.dropout(x)
    x = x + shortcut

    return x

In [98]:
class GPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.token_emb_layer = nn.Embedding(cfg['vocab_size'], cfg['emb_size'])
    # self.pos_emb_layer = nn.Embedding(cfg['context_length'], cfg['emb_size'])
    self.dropout_layer = nn.Dropout(cfg['drop_rate'])
    self.trf_blocks = nn.Sequential(
        *[TransformerBlock(cfg) for _ in range(cfg['num_layers'])]
    )
    self.final_norm = LayerNorm(cfg['emb_size'])
    self.output_layer = nn.Linear(cfg['emb_size'], cfg['vocab_size'], bias=False)

  def forward(self, inp_tokens):
    batch_size, num_tokens = inp_tokens.shape
    token_emb = self.token_emb_layer(inp_tokens)
    # pos_emb = self.pos_emb_layer(
    #     torch.arange(num_tokens, device=inp_tokens.device)
    # )
    # x = token_emb + pos_emb
    x = token_emb
    x = self.dropout_layer(x)
    x = self.trf_blocks(x)
    x = self.final_norm(x)
    logits = self.output_layer(x)

    return logits

In [99]:
def generate_text_tokens(model, inp_tokens, max_tokens, context_size):
  for _ in range(max_tokens):
    cropped_tokens = inp_tokens[:,-context_size:]
    with torch.no_grad():
      logits = model(cropped_tokens)

    logits = logits[:,-1,:]
    prob = torch.softmax(logits, dim=-1)
    token_id = torch.argmax(prob, dim=-1, keepdim=True)
    inp_tokens = torch.cat((inp_tokens, token_id), dim=1)

  return inp_tokens

In [100]:
def text_to_token_ids(text, tokenizer):
  encoded = tokenizer.encode(text)
  encoded_tensor = torch.tensor(encoded).unsqueeze(0)
  return encoded_tensor

def token_ids_to_text(tokens, tokenizer):
  token_list = tokens.squeeze(0).tolist()
  decoded = tokenizer.decode(token_list)
  return decoded

In [101]:
inp_text1 = "every effort moves"
inp_text2 = "I really like"

target_text1 = " effort moves you"
target_text2 = " really like chocolate"

In [102]:
tokenizer = tk.get_encoding("gpt2")

In [103]:
inp1 = text_to_token_ids(inp_text1, tokenizer)
inp2 = text_to_token_ids(inp_text2, tokenizer)

target1 = text_to_token_ids(target_text1, tokenizer)
target2 = text_to_token_ids(target_text2, tokenizer)

In [104]:
target1, target2

(tensor([[3626, 6100,  345]]), tensor([[ 1107,   588, 11311]]))

In [105]:
inp = torch.cat((inp1, inp2), dim=0)
target = torch.cat((target1 , target2), dim=0)

In [106]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()
with torch.no_grad():
  pred = model(inp)

print(pred.shape)

torch.Size([2, 3, 50257])


In [107]:
probs = torch.softmax(pred, dim=-1)
probs.shape

torch.Size([2, 3, 50257])

In [108]:
max_tokens = torch.argmax(probs, dim=-1, keepdim=True)
output_tokens = max_tokens
print(output_tokens)

tensor([[[35875],
         [40608],
         [35532]],

        [[27262],
         [18025],
         [17706]]])


In [109]:
print(f'Original text: {token_ids_to_text(target[0], tokenizer)}')
print(f'Predicted text: {token_ids_to_text(output_tokens[0].squeeze(-1), tokenizer)}')

Original text:  effort moves you
Predicted text: deck bitterly Partner


In [111]:
#probability score of target token ids
batch_target_idx = 0
prob_text1 = probs[batch_target_idx, [0,1,2], target[batch_target_idx]]

batch_target_idx = 1
prob_text2 = probs[batch_target_idx, [0,1,2], target[batch_target_idx]]

print(prob_text1)
print(prob_text2)

tensor([1.2436e-05, 2.2118e-05, 3.6531e-05])
tensor([3.0845e-05, 1.4014e-05, 1.0059e-05])


In [112]:
#LETS CALCULATE TEXT LOSS
##since we have already selected the probabilities of token indices
##first lets convert the probabilities to log probs and concatenate them
log_prob = torch.log(torch.cat((prob_text1, prob_text2), dim=-1))

##lets take average now
avg_log_prob = torch.mean(log_prob)

loss = avg_log_prob * -1

loss

tensor(10.8834)

In [113]:
#lets use cross_entropy funcntion to perform the above steps
pred_flat = torch.flatten(pred, 0, 1)
target_flat = torch.flatten(target)

pred_flat.shape, target_flat.shape

(torch.Size([6, 50257]), torch.Size([6]))

In [114]:
cross_entropy_loss = torch.nn.functional.cross_entropy(pred_flat, target_flat)
cross_entropy_loss

tensor(10.8834)

In [115]:
#lets calculate perplexity
perplexity = torch.exp(cross_entropy_loss)
perplexity

tensor(53284.2031)

In [116]:
class GPTDataset(Dataset):
  def __init__(self, text, tokenizer, max_length, stride):
    self.inputs = []
    self.targets = []

    encoded_text = tokenizer.encode(text)

    for i in range(0,len(encoded_text) - max_length, stride):
      input = encoded_text[i:i+max_length]
      target = encoded_text[i+1:i+max_length+1]

      self.inputs.append(torch.tensor(input))
      self.targets.append(torch.tensor(target))

  def __len__(self):
    return len(self.inputs)

  def __getitem__(self, index):
    return self.inputs[index], self.targets[index]

In [117]:
def create_dataloader(text, batch_size=8, max_length=4, stride=4, drop_last=True, num_workers=0, shuffle=True):
  tokenizer = tk.get_encoding('gpt2')
  dataset = GPTDataset(text,tokenizer, max_length, stride)
  print(len(dataset))

  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

  return dataloader

In [118]:
file_path = "/content/drive/My Drive/LLM/Data/the-verdict.txt"
with open(file_path,"r",encoding="utf-8") as f:
  raw_text = f.read()

print(len(raw_text))

20479


In [119]:
split_ratio = 0.9
train_text = raw_text[ : int(split_ratio * len(raw_text))]
val_text = raw_text[int(split_ratio * len(raw_text)) : ]

In [120]:
train_loader = create_dataloader(train_text, batch_size=2, max_length=GPT_CONFIG_124M['context_length'], stride=GPT_CONFIG_124M['context_length'])
val_loader = create_dataloader(val_text, batch_size=2, max_length=GPT_CONFIG_124M['context_length'], stride=GPT_CONFIG_124M['context_length'])

18
2


In [121]:
for x,y in train_loader:
  print(x.shape, y.shape)

torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])


In [122]:
for x,y in val_loader:
  print(x.shape, y.shape)

torch.Size([2, 256]) torch.Size([2, 256])


In [123]:
def ce_batch_loss_calc(input_batch, target_batch, model, device):
  """
  ARGS
  input_batch: torch.tensor 2-D
  target_batch: torch.tensor 2-D
  model: GPTModel
  device: torch.device - 'cuda' or 'cpu'

  calculates cross-entropy loss for a batch
  """
  input_batch = input_batch.to(device)
  target_batch = target_batch.to(device)
  logits_batch = model(input_batch)
  loss = torch.nn.functional.cross_entropy(logits_batch.flatten(0, 1), target_batch.flatten())

  return loss

In [124]:
def calc_loss_dataloader(loader, model, device, num_batches=None):
  """
  ARGS
  loader: dataloader
  model: GPTModel
  device: torch.device - 'cuda' or 'cpu'
  num_batches: integer

  calculates mean cross entropy loss across all the batches of the dataloader
  """
  total_loss = 0
  if len(loader) == 0:
    return float('nan')
  elif num_batches is None:
    num_batches = len(loader)
  elif num_batches < 0:
    num_batches = float('nan')
  else:
    num_batches = min(num_batches, len(loader))

  for i, (x, y) in enumerate(loader):
    if i < num_batches:
      loss = ce_batch_loss_calc(x, y, model, device)
      total_loss += loss
    else:
      break

  return total_loss/num_batches

In [125]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [126]:
model.to(device)
with torch.no_grad():
  train_loss = calc_loss_dataloader(train_loader, model=model, device = device)
  val_loss = calc_loss_dataloader(val_loader, model=model, device = device)

print(f'Train loss: {train_loss}')
print(f'Validation loss: {val_loss}')

Train loss: 10.97786808013916
Validation loss: 10.94667911529541


In [127]:
def train_model_simple(model, train_loader, val_loader, device, optimizer, epochs, eval_freq, eval_iter, start_context, tokenizer):
  train_loss_arr, val_loss_arr, track_tokens_seen = [], [], []
  global_step, tokens_seen = -1, 0

  for epoch in range(epochs):
    model.train()
    for x, y in train_loader:
      model.zero_grad()
      train_loss = ce_batch_loss_calc(x, y, model, device)
      train_loss.backward()
      optimizer.step()
      tokens_seen += x.numel()
      global_step += 1

      if global_step % eval_freq == 0:
        train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
        train_loss_arr.append(train_loss)
        val_loss_arr.append(val_loss)
        track_tokens_seen.append(tokens_seen)
        print(f'Train loss after epoch {epoch} (Step: {global_step}): {train_loss:.3f}')
        print(f'Val loss after epoch {epoch} (Step: {global_step}): {val_loss:.3f}')
        print(f'Number of tokens seen after epoch {epoch} (Step: {global_step})')

    generate_and_print_sample(model, start_context, tokenizer, device)

  return train_loss_arr, val_loss_arr, track_tokens_seen

In [128]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter=None):
  model.eval()
  with torch.no_grad():
    train_loss = calc_loss_dataloader(train_loader, model, device, eval_iter)
    val_loss = calc_loss_dataloader(val_loader, model, device, eval_iter)
  model.train()

  return train_loss, val_loss

In [129]:
def generate_and_print_sample(model, start_context, tokenizer, device):
  model.eval()
  context_size = model.trf_blocks[0].mha.mask.shape[0] ##model.pos_emb.shape[0]
  encoded_text = text_to_token_ids(start_context, tokenizer).to(device)
  with torch.no_grad():
    op_tokens = generate_text_tokens(model, encoded_text, max_tokens=50, context_size=context_size)
  decoded_text = token_ids_to_text(op_tokens, tokenizer)

  print(decoded_text.replace('\n',' '))
  model.train()

In [130]:
import time

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

t1 = time.time()
train_losses, val_losses, tokens_seen = train_model_simple(model, train_loader, val_loader, device, optimizer, epochs=10, eval_freq=5, eval_iter=5, start_context='Every effort moves you', tokenizer=tokenizer)
t2 = time.time()

print(f"Training time: {t2-t1}s")

Train loss after epoch 0 (Step: 0): 9.701
Val loss after epoch 0 (Step: 0): 9.938
Number of tokens seen after epoch 0 (Step: 0)
Train loss after epoch 0 (Step: 5): 7.933
Val loss after epoch 0 (Step: 5): 8.226
Number of tokens seen after epoch 0 (Step: 5)
Every effort moves you the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the
Train loss after epoch 1 (Step: 10): 6.501
Val loss after epoch 1 (Step: 10): 6.925
Number of tokens seen after epoch 1 (Step: 10)
Train loss after epoch 1 (Step: 15): 5.795
Val loss after epoch 1 (Step: 15): 6.467
Number of tokens seen after epoch 1 (Step: 15)
Every effort moves you of the the the the the the the the the of the the the of the the the of the the the of the the the of the the of the the the of the the of the the the of the of the the of the the the of
Train loss after epoch 2 (Step: 20): 5.260
Val lo

In [132]:
train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device)

print(f"Validation loss: {val_loss:.3f}")

Validation loss: 6.521


In [152]:
input = " Every efforts moves you in a way of Every efforts"
inp_tokens = text_to_token_ids(input, tokenizer).to(device)
print(inp_tokens)

embedding_layer = model.token_emb_layer
q_proj1 = model.trf_blocks[0].mha.w_query
k_proj1 = model.trf_blocks[0].mha.w_key

with torch.no_grad():
  embs = embedding_layer(inp_tokens)
  batch_size, seq_len, emb_size = embs.shape
  queries = q_proj1(embs)
  queries = q_proj1(embs).view(batch_size, seq_len, GPT_CONFIG_124M['num_heads'], GPT_CONFIG_124M['emb_size']//GPT_CONFIG_124M['num_heads']).transpose(1, 2)
  keys = k_proj1(embs).view(batch_size, seq_len, GPT_CONFIG_124M['num_heads'], GPT_CONFIG_124M['emb_size']//GPT_CONFIG_124M['num_heads']).transpose(1, 2)

  cos, sin = rotary_emb(GPT_CONFIG_124M['emb_size']//GPT_CONFIG_124M['num_heads'], seq_len, device)
  rotated_queries, rotated_keys = apply_rotation(queries, keys, cos, sin)

  q_cos_theta = torch.clamp(torch.sum((queries * rotated_queries), dim=-1)/(torch.norm(queries, dim=-1)*torch.norm(rotated_queries, dim=-1)), -1, 1)
  q_angles = torch.rad2deg(torch.acos(q_cos_theta))
  print(f"Query and rotated query angles:\n{q_angles}")

  k_cos_theta = torch.clamp(torch.sum((keys * rotated_keys), dim=-1)/(torch.norm(keys, dim=-1)*torch.norm(rotated_keys, dim=-1)), -1, 1)
  k_angles = torch.rad2deg(torch.acos(q_cos_theta))
  print(f"Key and rotated key angles:\n{k_angles}")

  # lets calculate attention score between "Every" and "efforts" both at start and end
  attention_score = (rotated_queries @ rotated_keys.transpose(-2, -1))
  print(f"first:\n {attention_score[...,0,1]}")
  print(f"second:\n {attention_score[...,-2,-1]}")
  # print(attention_score)

tensor([[3887, 4040, 6100,  345,  287,  257,  835,  286, 3887, 4040]],
       device='cuda:0')
Query and rotated query angles:
tensor([[[1.9782e-02, 1.7365e+01, 2.5681e+01, 2.3833e+01, 5.0536e+01,
          3.8876e+01, 3.6517e+01, 4.0776e+01, 4.9439e+01, 4.8676e+01],
         [0.0000e+00, 1.1528e+01, 2.6021e+01, 3.3571e+01, 4.8346e+01,
          4.6066e+01, 3.8466e+01, 5.9453e+01, 3.4213e+01, 5.6111e+01],
         [0.0000e+00, 1.6822e+01, 1.7115e+01, 4.8834e+01, 4.3234e+01,
          3.8868e+01, 3.6007e+01, 5.1224e+01, 5.1442e+01, 6.7170e+01],
         [1.9782e-02, 1.5761e+01, 1.9584e+01, 3.5202e+01, 4.8590e+01,
          4.9861e+01, 4.7329e+01, 3.9929e+01, 5.6885e+01, 6.1085e+01],
         [0.0000e+00, 1.6398e+01, 2.9899e+01, 4.6718e+01, 4.4986e+01,
          5.1462e+01, 3.7187e+01, 4.3447e+01, 4.5301e+01, 5.0991e+01],
         [0.0000e+00, 1.5195e+01, 3.0132e+01, 3.7168e+01, 4.1726e+01,
          3.0419e+01, 5.1452e+01, 3.8811e+01, 5.2243e+01, 4.7970e+01],
         [0.0000e+00, 1.745

The angle between query and rotated query is equal to angle between key and rotated key.

The attention score between "Every" query and "efforts" key is same irrespective of the absolute position and respects relative position.